In [1]:
print("HELLO")

HELLO


In [4]:
from langgraph.graph import START, END, StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, interrupt
from langchain_groq import ChatGroq
from langchain_classic.prompts import PromptTemplate
import os
from dotenv import load_dotenv
load_dotenv()

key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model="openai/gpt-oss-120b", api_key=key)



In [ ]:
from typing import List, Annotated
from pydantic import BaseModel
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, AIMessage, SystemMessage, HumanMessage
from pydantic import Field


class UserDetails(BaseModel):
    name: str
    lat: float
    long: float
    age: int
    language: str
    profession: str


class OnboardingState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    profile: UserDetails
    language: str


class InterviewState(BaseModel):
    passes: int
    total_score: int
    inst_score: int
    question: str
    answer: str
    messages: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    summary: str


In [25]:
from typing import Annotated
from pydantic import BaseModel, Field
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate

class UserDetails(BaseModel):
    name: str
    lat: float
    long: float
    age: int
    language: str
    profession: str



class InterviewState(BaseModel):
    passes: int = 0
    total_score: int = 0
    inst_score: int = 0
    user: UserDetails
    question: str = ""
    answer: str = ""

    messages: Annotated[list[BaseMessage], add_messages] = Field(
        default_factory=list
    )

    summary: str = ""


class Evaluator(BaseModel):
    score: float=Field(description="Float score from 1-10 based on how good the user's response to the question is.")

def ask_question(state: InterviewState):

    prompt = PromptTemplate.from_template("""
You are a conversational onboarding agent for a local employment-matching system.

Your task is to generate **the next best question** to ask a local worker based on the conversation so far.

The worker's basic information such as **name, age, profession, location, language, and other demographic details may already be available**. Do not ask for information that has already been provided.

You will receive the **previous conversation/messages as reference**. Carefully analyze them before generating the next question.

### Your Goal

Gradually understand the worker's **actual work experience, practical skills, responsibilities, methods, tools, strengths, and capabilities** so the information can later be used to match them with suitable employment opportunities.

### Rules

1. Ask **only ONE question at a time**.
2. The question must be **directly relevant to the previous answers**.
3. Prefer follow-up questions that dig deeper into something the worker has already mentioned.
4. Do not repeat questions or ask for information already present in the conversation.
5. If the worker mentions a specific job, task, tool, skill, or experience, use it to formulate a more specific follow-up.
6. Focus on **what they actually did and how they did it**, rather than generic questions about their skills.
7. Keep questions **short, simple, and conversational**, suitable for a local worker.
8. Avoid technical, corporate, or complicated language.
9. Prioritize information useful for **job matching**, such as:

   * Previous work performed
   * Specific responsibilities
   * Tasks they can independently handle
   * Tools or equipment they have used
   * Techniques or methods they know
   * Problems they have solved
   * Experience with different types of work
   * Strengths demonstrated through real work
10. Do not ask multiple questions in one message.
11. If the previous conversation already contains sufficient information about a topic, move to the next most useful missing area.
12. Never invent facts about the worker.
13. The conversation should feel like a **natural interview**, not a questionnaire.

### Input

You will receive the previous conversation/messages as context.
message history:
{messages}

user's name:
{name}

User's Profession:
{profession}

User's age:
{age}
### Output

Return **only the next question** to ask the worker.

Do not provide explanations, analysis, question numbers, or multiple alternatives.

""")
    chain = prompt | llm | StrOutputParser()
    question = chain.invoke({"messages":state.messages, "profession":state.user.profession, "age":state.user.age, "name":state.user.name})

    return {
        "question": question,
        "messages": [
            AIMessage(content=question)
        ]
    }


def get_answer(state: InterviewState):

    print("\nQuestion:", state.question)

    answer = input("Answer: ")

    return {
        "answer": answer,
        "messages": [
            HumanMessage(content=answer)
        ]
    }


def evaluate_answer(state: InterviewState):

    print("Evaluating:", state.answer)

    answer = state.answer
    question = state.question
    evaluator_prompt = PromptTemplate.from_template("""
You are an onboarding evaluator agent. Your task is to score the user's answer based on relevance and accuracy.
Question: {question}
\n
Answer: {answer}
""")

    structured_llm = llm.with_structured_output(Evaluator)
    chain = evaluator_prompt | structured_llm 
    score = chain.invoke({
        "answer":answer,
        "question":question
    })

    passes = state.passes + 1

    return {
        "inst_score": score.score,
        "total_score": state.total_score + score.score,
        "passes": passes
    }


def check_passes(state: InterviewState):

    if state.passes >= 3:
        return "end"

    return "continue"


builder = StateGraph(InterviewState)

builder.add_node("ask_question", ask_question)
builder.add_node("get_answer", get_answer)
builder.add_node("evaluate_answer", evaluate_answer)

builder.add_edge(START, "ask_question")
builder.add_edge("ask_question", "get_answer")
builder.add_edge("get_answer", "evaluate_answer")

builder.add_conditional_edges(
    "evaluate_answer",
    check_passes,
    {
        "continue": "ask_question",
        "end": END
    }
)

graph = builder.compile()

In [26]:
user = {
    "name":"Tanishq",
    "lat":24,
    "long":18,
    "age":21,
    "language":"hi",
    "profession":"Gaming Truck driver"
}



m1 = [SystemMessage(content="This is the start of onboarding system.")]
graph.invoke({
    "passes":0,
    "total_score":0,
    "inst_score":0,
    "user":user,
    "question":"",
    "answer":"",
    "messages":m1,
    "summary":""
    
})


Question: What are the main tasks you handle when you’re driving and setting up the gaming truck?
Evaluating: logistics

Question: Can you walk me through how you load, transport, and set up the gaming equipment for each event?
Evaluating: get and send

Question: When you arrive at a venue, what exact steps do you follow to set up the gaming stations and get everything running?
Evaluating: thats when its done 


{'passes': 3,
 'total_score': 4.0,
 'inst_score': 1.0,
 'user': {'name': 'Tanishq',
  'lat': 24,
  'long': 18,
  'age': 21,
  'language': 'hi',
  'profession': 'Gaming Truck driver'},
 'question': 'When you arrive at a venue, what exact steps do you follow to set up the gaming stations and get everything running?',
 'answer': 'thats when its done ',
 'messages': [SystemMessage(content='This is the start of onboarding system.', additional_kwargs={}, response_metadata={}, id='d6c0104a-4b21-4d38-a7af-958e5ecc7110'),
  AIMessage(content='What are the main tasks you handle when you’re driving and setting up the gaming truck?', additional_kwargs={}, response_metadata={}, id='2bc4e431-733c-4de1-8996-b6f2475dc270', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='logistics', additional_kwargs={}, response_metadata={}, id='c7b457d5-26dc-402e-9a64-3f5e8ee4e585'),
  AIMessage(content='Can you walk me through how you load, transport, and set up the gaming equipment for each event?', 

In [27]:
graph = StateGraph(InterviewState)



In [42]:


class ProfessionConfidence(BaseModel):
    confidence: float = Field(description="How well the LLM(you) understand the given profession")



def profession_confidence(profession):
    confidence_prompt = PromptTemplate.from_template("""
You are part of a user onboarding system that evaluates how well a worker knows their own profession.

Before you can generate good questions, you must first judge yourself: do you know enough about this profession to ask specific, relevant experience and knowledge questions about it?

Profession: {profession}

### Instructions

1. Think about whether this profession involves tasks, tools, and terminology you're familiar with.
2. If it's a common, well-documented profession (e.g. electrician, driver, tailor, cook), you likely have strong knowledge.
3. If it's vague, highly niche, informal, or region-specific in a way you can't reliably reason about, your knowledge is weak.
4. Do not guess or pretend to know more than you do — an honest low score is better than a wrong high score, since it will change how the user is questioned.

### Output

Return a confidence score from 1-10 for how well you know this profession well enough to ask specific, meaningful questions about it.
""")
     
    structured_llm = llm.with_structured_output(ProfessionConfidence)
    chain = confidence_prompt | structured_llm
    score = chain.invoke({"profession":profession})
    return{
        "confidence":score.confidence
    }



    

In [47]:
profession_confidence("rally worker")

{'confidence': 5.0}

In [1]:
text = "The user is a professional electrician with hands‑on experience in residential wiring and fuse‑box repair. They regularly perform house wiring tasks, install and upgrade fuse boxes, and troubleshoot electrical issues using tools such as a multimeter, screwdriver set, and wire stripper. Notably, they resolved a short‑circuit problem that other electricians could not locate, demonstrating strong diagnostic skills and practical problem‑solving ability"

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
import os
from dotenv import load_dotenv

load_dotenv()

key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model="openai/gpt-oss-120b", api_key=key)

class keywordExtractor(BaseModel):
    keywords: list[str] = Field(description="Useful and crucial keywords to identify worker's qualities.")
def extractKeywords(text):
        prompt = PromptTemplate.from_template("""
You are a keyword extractor for a worker-matching platform. Extract concrete, searchable keywords from the worker's conversation below — things a job posting would match against.

CONVERSATION
{text}

INCLUDE: tasks performed, tools/equipment used, techniques demonstrated, specialization/domain — only if explicitly stated or clearly implied.
EXCLUDE: generic words (experience, skilled), invented capabilities, personal details (name, age, location).

Keywords: 1-4 words each, lowercase, no duplicates/near-duplicates.

Return ONLY the list of keywords.
""")
        structured_llm = llm.with_structured_output(keywordExtractor)
        chain = prompt | structured_llm

        keywords = chain.invoke({"text":text})
        return keywords
        


In [6]:
extractKeywords(text=text)

keywordExtractor(keywords=['electrician', 'residential wiring', 'fuse box repair', 'house wiring', 'fuse box installation', 'fuse box upgrade', 'electrical troubleshooting', 'multimeter', 'screwdriver set', 'wire stripper', 'short circuit diagnosis'])